# Proyecto 1: Búsqueda en Espacios de Estados
## El Puente y la Linterna (The Bridge and the Torch)

**Curso:** Inteligencia Artificial  
**Institución:** Universidad Michoacana de San Nicolás de Hidalgo  
**Profesor:** José Ortiz Béjar

**Integrantes:**
* José David Escobedo Villa (Matrícula: 1911663h)
* Edgar Alan Ocaña Lemus (Matrícula: 2201227f)

**Enlace al repositorio (Fork público):** https://github.com/David34uni/baile

## 1. Descripción del Problema
Un grupo de $N$ personas necesita cruzar un puente estrecho por la noche para llegar al otro lado de un río. 
El puente solo puede sostener a un máximo de **dos personas a la vez** y es obligatorio que cualquier grupo que lo cruce lleve una **linterna** para iluminar el camino.

Poseen una sola linterna, por lo que una vez que un grupo cruza a la orilla derecha, alguien debe regresar caminando con la linterna hacia la orilla izquierda para permitir que los demás sigan cruzando. 
Cada persona camina a una velocidad distinta, y cuando dos personas cruzan juntas, deben avanzar al ritmo de la persona **más lenta** del par.

**Objetivo:** Trasladar a todas las personas de la orilla izquierda a la derecha en el **menor tiempo total acumulado** (minutos).

## 2. Modelado Formal

1. **Estado:** Representado por la tupla `(frozenset(personas_izq), pos_linterna)`
   * `personas_izq`: `frozenset` con los tiempos individuales de las personas que siguen en la orilla izquierda.
   * `pos_linterna`: Entero `0` (orilla izquierda) o `1` (orilla derecha).
   * *Justificación:* Es un estado **mínimo y hashable**. La orilla derecha se deriva implícitamente del complemento ($S_{total} \setminus S_{izq}$). Omitir variables como el costo acumulado previene la duplicación inútil del espacio de estados.

2. **Operación Sucesor:**
   * Si la linterna está en `0`, se eligen 1 o 2 personas de la izquierda para cruzar a la derecha.
   * Si la linterna está en `1`, se eligen 1 o 2 personas de la derecha para regresar a la izquierda.
   * El costo de paso (`step_cost`) de cada movimiento es `max(tiempo_p1, tiempo_p2)`.

3. **Condición de Meta:**
   * `len(personas_izq) == 0` y `pos_linterna == 1`.

4. **Heurísticas:**
   * **$h_1$ (Candidata - Máximo):** $h_1(n) = \max(\text{tiempos en la izquierda})$. Admisible porque al menos la persona más lenta tendrá que cruzar el puente.
   * **$h_2$ (Secundaria - Promedio):** $h_2(n) = \text{promedio}(\text{tiempos en la izquierda})$.
   * **$h_0$ (Línea Base):** $h_0(n) = 0$ (Costo Uniforme).

## 3. Estimación del Tamaño de Búsqueda
* Factor de ramificación promedio: $b \approx 4.5$
* Profundidad típica para $N=4$: $d = 5$ movimientos.
* Árbol sin memoria: $b^d \approx 4.5^5 = 1,845$ nodos.
* Espacio de estados acotado: $2^{N+1} = 2^5 = 32$ estados posibles para $N=4$.

In [37]:
import sys
import os
import time
import pandas as pd

ruta_src = os.path.abspath("src")
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import SimpleSearch as sp
from puente_linterna import crear_sucesor, meta, h1_maximo, h2_promedio, h0_nula

print("Módulos cargados correctamente.")

Módulos cargados correctamente.


In [38]:
def ejecutar_experimento(instancias, max_iter=500000):
    resultados = []

    estrategias = [
        ("BFS", "bfs", None),
        ("DFS", "dfs", None),
        ("A* (h=0 / Costo Uniforme)", "a*", h0_nula),
        ("A* (h1 = Máximo)", "a*", h1_maximo),
        ("A* (h2 = Promedio)", "a*", h2_promedio),
    ]

    for nombre_instancia, tiempos in instancias.items():
        sucesor_fn = crear_sucesor(tiempos)
        estado_inicial = (frozenset(tiempos), 0)

        for nombre_est, strat, h_fn in estrategias:
            inicio_nodo = sp.Node(estado_inicial)

            if strat == "a*":
                b = sp.TreeSearch(
                    inicio_nodo, sucesor_fn, meta, strategy=strat, heuristic=h_fn
                )
            else:
                b = sp.TreeSearch(inicio_nodo, sucesor_fn, meta, strategy=strat)

            t0 = time.time()
            r = b.find(max_iter=max_iter)
            t1 = time.time()

            if r is not None:
                nodos = b.iterations
                tiempo_s = t1 - t0
                longitud = len(r.getPath()) - 1
                costo_g = r.cost
            else:
                nodos = b.iterations
                tiempo_s = t1 - t0
                longitud = "Sin solución"
                costo_g = "N/A"

            resultados.append(
                {
                    "Instancia": nombre_instancia,
                    "Estrategia": nombre_est,
                    "Nodos Expandidos": nodos,
                    "Tiempo (s)": round(tiempo_s, 5),
                    "Longitud (pasos)": longitud,
                    "Costo Total g(n)": costo_g,
                }
            )

    return pd.DataFrame(resultados)

In [39]:
instancias = {
    "Fácil (N=4)": [1, 2, 5, 10],
    "Media (N=5)": [1, 2, 4, 8, 12],
    "Difícil (N=6)": [1, 2, 5, 8, 10, 15],
}

df_resultados = ejecutar_experimento(instancias, max_iter=500000)
df_resultados

,Instancia,Estrategia,Nodos Expandidos,Tiempo (s),Longitud (pasos),Costo Total g(n)
0,Fácil (N=4),BFS,25,0.00043,5,19
1,Fácil (N=4),DFS,8,0.00019,7,50
2,Fácil (N=4),A* (h=0 / Costo Uniforme),25,0.00041,5,17
3,Fácil (N=4),A* (h1 = Máximo),18,0.00033,5,17
4,Fácil (N=4),A* (h2 = Promedio),18,0.00049,5,17
5,Media (N=5),BFS,56,0.00163,7,47
6,Media (N=5),DFS,15,0.00039,13,156
7,Media (N=5),A* (h=0 / Costo Uniforme),56,0.00116,7,24
8,Media (N=5),A* (h1 = Máximo),39,0.00092,7,24
9,Media (N=5),A* (h2 = Promedio),47,0.00114,7,24


## 4. Análisis de la Heurística y Verificación de Correctitud

1. **Verificación de Correctitud:**
   Dado que los costos de paso son no uniformes (`step_cost != 1`), la prueba de optimalidad se verifica comparando el costo total $g(n)$ obtenido por $A^*$ contra $A^*$ con $h(n)=0$ (Costo Uniforme). 
   En los resultados se comprueba que $A^*$ con $h_1$ obtiene exactamente el mismo costo $g(n)$ mínimo que $A^*$ con $h(n)=0$, demostrando la admisibilidad de $h_1$.

2. **Comparación BFS vs A*:**
   BFS busca optimizar el **número de pasos** (longitud del camino), pero no considera el costo no uniforme en minutos. Por lo tanto, BFS puede encontrar caminos de menor cantidad de movimientos pero con un costo total de tiempo superior. $A^*$ con $h_1$ encuentra el camino óptimo en minutos expandiendo significativamente menos nodos.

3. **Demostración de Admisibilidad de $h_1$ (Relajación):**
   Si se eliminan las restricciones del puente (capacidad máxima de 2 personas y necesidad de linterna) y se asume que todas las personas en la orilla izquierda pueden cruzar instantáneamente en un solo viaje, el tiempo mínimo que le tomará a la persona más lenta cruzar es $\max(\text{tiempos en izquierda})$. 
   Dado que el problema real jamás puede resolverse en menos tiempo que en este escenario relajado, $h_1(n) \le h^*(n)$ para todo estado $n$, garantizando admisibilidad.

## 5. Conclusiones
* $A^*$ con la heurística admisible $h_1$ superó a las búsquedas no informadas en eficiencia, alcanzando el costo mínimo óptimo en tiempo.
* BFS no garantiza la solución óptima en problemas con costos no uniformes.
* El modelado correcto con `frozenset` evitó explosiones combinatorias por estados repetidos.

### Contribución de Integrantes
* **José David Escobedo Villa:** Modelado del espacio de estados, función sucesor y desarrollo del módulo `puente_linterna.py`.
* **Edgar Alan Ocaña Lemus:** Implementación del protocolo experimental en Jupyter Notebook, análisis de admisibilidad de la heurística y redacción del reporte final.